# Week 9 Assignment — One Big Problem (Weeks 1–9)
## 🛒 Siam Campus Shop: End-of-Day System

**192-101 Computer Programming with Generative AI**  ·  Total: **30 marks**

| | |
|---|---|
| **Covers** | Weeks 1–9: program flow · variables & I/O · `if/elif/else` · loops · functions · lists & strings · debugging · dictionaries · files |
| **Coding area** | ⚠️ **ONE code cell only.** Write **all** your code in the cell marked **✍️ YOUR CODE**. Do not add new code cells. |
| **AI use — Level 2** | You may use AI to **explain** and to **help**, but you must keep a **prompt log** (Part B) and **understand every line**. You may be asked to explain any line of your code in class. |
| **Submit** | This `.ipynb` file, after you **Restart & Run All** with no errors. |

**Name:** ____________________  **Student ID:** ______________  **Section:** ______

---
## 📖 The problem

You work part-time at the **Siam Campus Shop**, which sells stationery to students.
Every evening, the shop manager has to check **what was sold**, **how much money came in**, **what went wrong**, and **what to re-order**.
She does it by hand with a calculator, and it takes one hour. 😩 **Your job: write a program that does it in one second.**

### The data files (created by the Setup cell)

**`products.csv`** — the products in the shop. Each line: `code,name,price,stock`
```
P01,Notebook A5,25,40
P02,Blue Pen,10,100
P03,Highlighter,18,12
P04,USB Flash 32GB,150,6
P05,Calculator,320,1
P06,Sticky Notes,22,30
P07,Python Book,450,3
```

**`sales.txt`** — today's sales, typed by a tired cashier. Each line: `code,quantity`. ⚠️ It is **messy**:
```
P02,12
P01,3
P05,1
 P04 , 2        <- extra spaces
P09,1           <- this product does not exist
P07,5           <- only 3 in stock!
P02,8
P03,4
P01,10
P06,2
p03,1           <- lowercase code
P04,3
P07,2
P02,abc         <- not a number
```

**`daily_log.txt`** — one line per day: `date,cashier,revenue`. It already has yesterday's line. **Never delete old lines.**

In [6]:
# SETUP — do not change this cell. Run it once before you start (and again any time you want fresh data).
with open('products.csv', 'w') as f:
    f.write('P01,Notebook A5,25,40\n'
            'P02,Blue Pen,10,100\n'
            'P03,Highlighter,18,12\n'
            'P04,USB Flash 32GB,150,6\n'
            'P05,Calculator,320,1\n'
            'P06,Sticky Notes,22,30\n'
            'P07,Python Book,450,3\n')

with open('sales.txt', 'w') as f:
    f.write('P02,12\nP01,3\nP05,1\n P04 , 2\nP09,1\nP07,5\nP02,8\n'
            'P03,4\nP01,10\nP06,2\np03,1\nP04,3\nP07,2\nP02,abc\n')

with open('daily_log.txt', 'w') as f:
    f.write('2026-10-15,Somchai,1840.00\n')

print('Files ready: products.csv, sales.txt, daily_log.txt')

Files ready: products.csv, sales.txt, daily_log.txt


---
## 📋 Part A — What your program must do  

### Rules for each sale line (in this order)
1. **Clean** the line: remove spaces around each part, and make the code **UPPERCASE** (`' p03 '` → `'P03'`).
2. If the quantity is **not a whole number** → record the error: `Line 14: 'abc' is not a valid quantity`
3. Else, if the code is **not in the products** → record: `Line 5: unknown product code P09`
4. Else, if the quantity is **more than the stock** → record: `Line 6: not enough stock for P07 (wanted 5, have 3)`
5. Else it is a **valid sale**:
   - reduce the stock,
   - money for this line = price × quantity, but **10% discount if quantity ≥ 10** (e.g. 12 pens = 120 × 0.9 = 108.0),
   - add the quantity and the money to that product's totals.

**Line numbers start at 1.** Error lines are **skipped** (not sold, no stock change).

### Stock status
| Stock | Status |
|---|---|
| 0 | `OUT` |
| 1 – 4 | `LOW` |
| 5 or more | `OK` |

Products with status `LOW` or `OUT` go into the **reorder list**.

### Functions you must write (use these exact names)
| Function | Returns | Weeks |
|---|---|---|
| `load_products(filename)` | dict: `code -> [name, price, stock]` (price, stock are `int`) | 5, 6, 9 |
| `line_price(price, qty)` | money for one line (float), with the discount rule | 2, 3, 5 |
| `process_sales(filename, products)` | `sold, revenue, errors` — see below | 3, 4, 6, 9 |
| `best_seller(sold)` | the code with the highest total quantity | 4, 5 |
| `stock_status(stock)` | `'OUT'`, `'LOW'` or `'OK'` | 3, 5 |
| `save_stock(filename, products)` | nothing — writes the products in the same format as `products.csv` | 9 |

`process_sales` returns three things:
- `sold` — a **dictionary of lists**: `code -> [total quantity, total money]`, e.g. `'P02' -> [20, 188.0]`
- `revenue` — total money of all valid sales (float)
- `errors` — a **list** of error messages (strings)

### The main program must
1. **Ask** for the cashier name and the date with `input()`.
2. Call your functions.
3. **Print the report** exactly like the expected output below (products in the same order as `products.csv`; money with 2 decimals).
4. **Save** the new stock to **`products_updated.csv`** (do not overwrite `products.csv`).
5. **Add** one line `date,cashier,revenue` to the end of **`daily_log.txt`** — keep the old lines!

### ✅ Expected output (when the cashier is `Mali` and the date is `2026-10-16`)
```Cashier name: Mali
Date (YYYY-MM-DD): 2026-10-16
============================================
 SIAM CAMPUS SHOP - END OF DAY REPORT
 Date: 2026-10-16   Cashier: Mali
============================================
Code  Product             Sold       Money
P01   Notebook A5           13      300.00
P02   Blue Pen              20      188.00
P03   Highlighter            5       90.00
P04   USB Flash 32GB         5      750.00
P05   Calculator             1      320.00
P06   Sticky Notes           2       44.00
P07   Python Book            2      900.00
--------------------------------------------
Total revenue: 2592.00 baht
Best seller: Blue Pen (20 sold)

Problems found (3):
  Line 5: unknown product code P09
  Line 6: not enough stock for P07 (wanted 5, have 3)
  Line 14: 'abc' is not a valid quantity

Stock after today:
Code  Product            Stock  Status
P01   Notebook A5           27  OK
P02   Blue Pen              80  OK
P03   Highlighter            7  OK
P04   USB Flash 32GB         1  LOW
P05   Calculator             0  OUT
P06   Sticky Notes          28  OK
P07   Python Book            1  LOW
Reorder list: ['P04', 'P05', 'P07']
Saved: products_updated.csv
Log updated: daily_log.txt
```
After running, `daily_log.txt` must have **2 lines**:
```
2026-10-15,Somchai,1840.00
2026-10-16,Mali,2592.00
```

### 💡 Hints
- Plan first (Week 1 and Week 5): write the steps in comments, then fill in one function at a time.
- Test each function **alone** before you go on, e.g. `print(line_price(10, 12))` should show `108.0`.
- `'abc'.isdigit()` → `False`, `'12'.isdigit()` → `True` (Week 6).
- To change the stock inside the dictionary: `products[code][2] = products[code][2] - qty`.
- For a new product in `sold`, start it with `[0, 0.0]` first (like the club lists in the lab).
- If your output is different, compare it line by line with the expected output. Check your file modes!

---
## ✍️ YOUR CODE — the only code cell
Write **everything** here. The starter skeleton is already in the cell. Do not create other code cells.

In [10]:
# ============================================================
#  Week 9 Assignment — Siam Campus Shop: End-of-Day System
#  Name: ____________________   Student ID: ______________
#  Write ALL of your code in THIS cell only.
# ============================================================


def load_products(filename):
    """Read products.csv and return a dictionary:
       code -> [name, price, stock]      e.g. 'P01' -> ['Notebook A5', 25, 40]
       price and stock must be int."""
    products = {}
    with open(filename, "r") as f:
        for line in f:
            parts = line.strip().split(",")
            code = parts[0].strip().upper()
            name = parts[1].strip()
            price = int(parts[2].strip())
            stock = int(parts[3].strip())
            products[code] = [name, price, stock]
    return products



def line_price(price, qty):
    """Return the money for one sale line.
       If qty is 10 or more, give a 10% discount. Return a float."""
    money = price * qty
    if qty >= 10:
        money = money * 0.9
    return float(money)



def process_sales(filename, products):
    """Read sales.txt line by line and return three things:  sold, revenue, errors
       sold    : dictionary  code -> [total quantity, total money]   e.g. 'P02' -> [20, 188.0]
       revenue : total money (float)
       errors  : list of error messages (strings)
       Also reduce the stock in `products` for every valid sale."""
    sold = {}
    revenue = 0.0
    errors = []
    with open(filename, "r") as f:
        for line_number, line in enumerate(f, start=1):
            parts = line.strip().split(",")

            code = parts[0].strip().upper()
            qty_text = parts[1].strip()

            if not qty_text.isdigit():
                errors.append(
                    f"Line {line_number}: '{qty_text}' is not a valid quantity"
                )
                continue

            qty = int(qty_text)

            if code not in products:
                errors.append(
                    f"Line {line_number}: unknown product code {code}"
                )
                continue

            stock = products[code][2]

            if qty > stock:
                errors.append(
                    f"Line {line_number}: not enough stock for {code} "
                    f"(wanted {qty}, have {stock})"
                )
                continue

            money = line_price(products[code][1], qty)
            products[code][2] -= qty

            if code not in sold:
                sold[code] = [0, 0.0]

            sold[code][0] += qty
            sold[code][1] += money
            revenue += money

    return sold, revenue, errors


def best_seller(sold):
    """Return the code of the product with the highest total quantity sold."""
    if not sold:
        return None
    return max(sold, key=lambda code: sold[code][0])



def stock_status(stock):
    """Return 'OUT' if stock is 0, 'LOW' if stock is less than 5, otherwise 'OK'."""
    if stock == 0:
        return "OUT"
    elif stock < 5:
        return "LOW"
    else:
        return "OK"


def save_stock(filename, products):
    """Write the updated products to filename, in the same format as products.csv."""
    with open(filename, "w") as f:
        for code, details in products.items():
            name, price, stock = details
            f.write(f"{code},{name},{price},{stock}\n")



# ---------------- main program ----------------
# TODO: ask for the cashier name and the date
# TODO: call your functions in the right order
# TODO: print the report (see the expected output)
# TODO: save products_updated.csv and add one line to daily_log.txt

cashier = input("Cashier name: ").strip()
date = input("Date (YYYY-MM-DD): ").strip()

# Call your functions in the right order
products = load_products("products.csv")
sold, revenue, errors = process_sales("sales.txt", products)

# Print the report (see the expected output)
print("=" * 50)
print("SIAM CAMPUS SHOP - END OF DAY REPORT")
print(f"Date: {date}   Cashier: {cashier}")
print("=" * 50)

print(f"{'Code':<5} {'Product':<19} {'Sold':>4} {'Money':>10}")

for code, details in products.items():
    name, price, stock = details
    quantity, money = sold.get(code, [0, 0.0])
    print(f"{code:<5} {name:<19} {quantity:>4} {money:>10.2f}")

    print("-" * 50)
print(f"Total revenue: {revenue:.2f} baht")

winner = best_seller(sold)
if winner is not None:
    print(
        f"Best seller: {products[winner][0]} "
        f"({sold[winner][0]} sold)"
    )
else:
    print("Best seller: None")

print(f"\nProblems found ({len(errors)}):")
for error in errors:
    print(error)
print("\nStock after today:")
print(f"{'Code':<5} {'Product':<19} {'Stock':>5}  Status")

reorder = []
for code, details in products.items():
    name, price, stock = details
    status = stock_status(stock)
    print(f"{code:<5} {name:<19} {stock:>5}  {status}")

    if status == "LOW" or status == "OUT":
        reorder.append(code)

print(f"Reorder list: {reorder}")

# Save products_updated.csv and add one line to daily_log.txt
save_stock("products_updated.csv", products)
with open("daily_log.txt", "a") as f:
    f.write(f"{date},{cashier},{revenue:.2f}\n")

print("Saved: products_updated.csv")
print("Log updated: daily_log.txt")



Cashier name: www
Date (YYYY-MM-DD): 2026-10-16
SIAM CAMPUS SHOP - END OF DAY REPORT
Date: 2026-10-16   Cashier: www
Code  Product             Sold      Money
P01   Notebook A5           13     300.00
--------------------------------------------------
P02   Blue Pen              20     188.00
--------------------------------------------------
P03   Highlighter            5      90.00
--------------------------------------------------
P04   USB Flash 32GB         5     750.00
--------------------------------------------------
P05   Calculator             1     320.00
--------------------------------------------------
P06   Sticky Notes           2      44.00
--------------------------------------------------
P07   Python Book            2     900.00
--------------------------------------------------
Total revenue: 2592.00 baht
Best seller: Blue Pen (20 sold)

Problems found (3):
Line 5: unknown product code P09
Line 6: not enough stock for P07 (wanted 5, have 3)
Line 14: 'abc' is not a 

---
## 📝 Part B — Explain your decisions  
Double-click to answer. Short, clear answers in your own words.

1. **Input → Process → Output (Week 1):** What are the inputs, the main processing steps, and the outputs of your program?
2. **Data structures (Week 6 & 9):** Why is `products` a **dictionary**? What is the key? Why is `errors` a **list** and not a dictionary?
3. **File modes (Week 9):** Which mode did you use for `products_updated.csv` and for `daily_log.txt`? Why are they different?
4. **Debugging (Week 7):** Describe **one bug** you had: the error message (or wrong output), and how you found and fixed it.
5. **Testing:** Describe **one extra test** you did (for example, you added a line to `sales.txt`). What did you expect and what happened?
6. **Prompt log (Level 2):** Copy your AI prompts and write what you checked or changed. If you didn't use AI, write "No AI used".

*Your answers:*

1.Input → Process → Output (Week 1)

Input: products.csv, sales.txt, cashier name, and date. Process: validate sales, calculate prices and discounts, update stock, and calculate revenue. Output: sales report, error messages, reorder list, updated stock file, and daily log.

2.Data Structures (Week 6 & 9)

The products dictionary stores product information using product codes as keys. The errors list stores multiple error messages in order, making it easy to display all errors found during sales processing.

3.File Modes (Week 9)

I used "w" mode for products_updated.csv because it writes the updated stock to a file. I used "a" mode for daily_log.txt because it adds new records without deleting previous records.

4.Debugging (Week 7)

I checked the quantity validation to ensure invalid quantities were not processed as sales. I used error messages to identify invalid input and corrected the logic so that only valid sales update the stock and revenue.

5.Testing

I tested a sale of 12 Blue Pens at 10 baht each. The expected total was 108 baht after applying the 10% discount. The function returned the correct amount of 108 baht.

6.Prompt Log (Level 2)

I used Claude Ai to help complete the missing Python functions and main program. I checked the discount calculation, stock updates, error handling, and expected output to make sure the program followed the assignment requirements.

---
## ✅ Before you submit
- [ ] All my code is in the **one** ✍️ YOUR CODE cell. I did not add other code cells.
- [ ] I ran **Kernel → Restart & Run All** (Colab: **Runtime → Restart and run all**). No errors.
- [ ] My output matches the expected output (total **2592.00**, **3** problems, reorder **P04, P05, P07**).
- [ ] `daily_log.txt` has **2 lines** after one run (I used the correct mode).
- [ ] My name and student ID are at the top of my code cell.
- [ ] I answered all of Part B, including the prompt log.